# SPX 0DTE captured net-premium divergence audit

## tl;dr

主规则在15分钟上有 229 个熊背离事件，平均做空结果 0.06 点，session-block 95%区间 [-0.45, 1.59]；首个早盘信号配对交易 27 笔，平均 -0.80 点。单独限制10:00 ET以后，熊背离60分钟平均 1.40 点、120分钟平均 0.56 点；首个信号配对平均 -0.13 点。没有同时通过置信区间、安慰剂和配对策略门；不应接入生产决策。

## Context & Methods

### Key Assumptions

- 只把最新成交在 ask 及以上视为买入、bid 及以下视为卖出；价差内部成交不猜方向。
- 熊背离：最近5分钟创新高后回落，同时 Call net≤0、Put net>0。
- 牛背离/空头退出：最近5分钟创新低后回升，同时 Call net≥0、Put net≤0。
- 信号在分钟结束后才成立；8月26日只作截图复现，不进入历史统计。
- 结果是SPX点数，不是假装可成交的期权PnL。

## Data

请求范围 2026-07-07 至 2026-08-26，实际首个Schwab期权日为 2026-07-13；发现 33 个工作日，32 个完整SPX日，32 个达到最低成交覆盖门。原始源为 Schwab L1 snapshot lake，不是完整OPRA tape。

In [1]:
from pathlib import Path
import json

repo = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src/spx_spark').is_dir())
artifact = repo / 'docs/research/net-premium-divergence-backtest-2026-08-27.json'
analysis = json.loads(artifact.read_text())
analysis['data_quality']['summary']

{'active_flow_minute_rate': {'maximum': 1.0,
  'median': 1.0,
  'minimum': 0.33240997229916897},
 'at_touch_volume_coverage': {'maximum': 0.16391815258696288,
  'median': 0.1381564631204656,
  'minimum': 0.10725982131147016},
 'classified_premium_share': {'maximum': 0.7293510749625183,
  'median': 0.6313947859904306,
  'minimum': 0.4314440030268415},
 'complete_spx_sessions': 32,
 'inside_or_unclassified_share': {'maximum': 0.4487913809990206,
  'median': 0.4027878708701107,
  'minimum': 0.3402751756913762},
 'minimum_coverage_sessions': 32,
 'sessions': 33,
 'tape_capture_ratio': {'maximum': 0.2969798304589301,
  'median': 0.23679396192591598,
  'minimum': 0.1819027481438593}}

## Results

In [2]:
for period, result in analysis['primary_period_results'].items():
    bear = result['bearish_divergence']['forward_15m']
    base = result['price_high_failure']['forward_15m']
    bull = result['bullish_exhaustion']['forward_15m']
    print(period, {'bear_n': bear['n'], 'bear_mean': bear['mean_points'], 'bear_ci': bear['session_block_ci95_mean_points'], 'price_only_mean': base['mean_points'], 'bull_exit_rebound': bull['mean_points']})

all {'bear_n': 229, 'bear_mean': 0.06227074235806764, 'bear_ci': [-0.44617696552999103, 1.589032849520544], 'price_only_mean': -0.5872451790633686, 'bull_exit_rebound': -0.5327230046948326}
development {'bear_n': 103, 'bear_mean': -0.3023300970873995, 'bear_ci': [-1.373665578403113, 2.280722300384717], 'price_only_mean': -0.8548235294117683, 'bull_exit_rebound': -1.050689655172446}
tail {'bear_n': 40, 'bear_mean': 1.8379999999999654, 'bear_ci': [0.9342499999999859, 2.841535714285756], 'price_only_mean': 0.408676470588208, 'bull_exit_rebound': 0.12085106382979342}
validation {'bear_n': 86, 'bear_mean': -0.3269767441860347, 'bear_ci': [-1.1169395847900832, 1.2603639339826798], 'price_only_mean': -0.7651200000000026, 'bull_exit_rebound': 0.05460000000008222}


In [3]:
analysis['paired_first_morning_short']['periods']

{'all': {'bootstrap_ci95_mean_gross_points': [-5.435527777777743,
   3.601583333333356],
  'cost_sensitivity': {'roundtrip_0.0_points': {'mean_net_points': -0.7962962962962626,
    'total_net_points': -21.49999999999909,
    'win_rate': 0.5555555555555556},
   'roundtrip_0.5_points': {'mean_net_points': -1.2962962962962625,
    'total_net_points': -34.99999999999909,
    'win_rate': 0.5555555555555556},
   'roundtrip_1.0_points': {'mean_net_points': -1.7962962962962625,
    'total_net_points': -48.49999999999909,
    'win_rate': 0.5555555555555556}},
  'gross_max_drawdown_points': 85.19999999999982,
  'gross_win_rate': 0.5555555555555556,
  'mean_gross_points': -0.7962962962962626,
  'median_gross_points': 3.6099999999996726,
  'signal_exits': 27,
  'trades': 27},
 'development': {'bootstrap_ci95_mean_gross_points': [-8.366812499999892,
   5.665145833333521],
  'cost_sensitivity': {'roundtrip_0.0_points': {'mean_net_points': -1.051666666666506,
    'total_net_points': -12.6199999999980

In [4]:
after = analysis['after_1000_et_extension']
for kind, summary in after['event_summaries'].items():
    print(kind, {key: value['mean_points'] for key, value in summary.items() if isinstance(value, dict) and 'mean_points' in value})
after['first_signal_trade']['periods']

bearish_divergence {'forward_120m': 0.5565644171778934, 'forward_15m': 0.14022321428570553, 'forward_30m': 0.610758928571406, 'forward_5m': 0.32607142857140403, 'forward_60m': 1.3961083743842357, 'to_1545': 2.7465625000000022}
bullish_exhaustion {'forward_120m': -0.709530201342247, 'forward_15m': -0.6743478260869529, 'forward_30m': -0.6614563106796215, 'forward_5m': -0.18106280193234306, 'forward_60m': -0.8747849462365789, 'to_1545': -4.88183574879235}
price_high_failure {'forward_120m': -0.727346153846198, 'forward_15m': -0.4150704225352174, 'forward_30m': -0.08677053824363411, 'forward_5m': -0.10391549295775765, 'forward_60m': 0.2818575851393357, 'to_1545': 1.9173521126760917}
price_low_failure {'forward_120m': -0.574765625000019, 'forward_15m': -0.58735211267608, 'forward_30m': -0.4615099715099683, 'forward_5m': -0.12177464788731503, 'forward_60m': -0.24906542056076694, 'to_1545': -4.299464788732462}


{'all': {'bootstrap_ci95_mean_gross_points': [-4.393091666666603,
   3.804700000000042],
  'cost_sensitivity': {'roundtrip_0.0_points': {'mean_net_points': -0.13499999999994544,
    'total_net_points': -4.049999999998363,
    'win_rate': 0.6},
   'roundtrip_0.5_points': {'mean_net_points': -0.6349999999999454,
    'total_net_points': -19.049999999998363,
    'win_rate': 0.6},
   'roundtrip_1.0_points': {'mean_net_points': -1.1349999999999454,
    'total_net_points': -34.04999999999836,
    'win_rate': 0.6}},
  'gross_max_drawdown_points': 68.35999999999876,
  'gross_win_rate': 0.6,
  'mean_gross_points': -0.13499999999994544,
  'median_gross_points': 4.079999999999927,
  'signal_exits': 30,
  'trades': 30},
 'development': {'bootstrap_ci95_mean_gross_points': [-8.007178571428463,
   5.991553571428642],
  'cost_sensitivity': {'roundtrip_0.0_points': {'mean_net_points': -0.8757142857141714,
    'total_net_points': -12.2599999999984,
    'win_rate': 0.5714285714285714},
   'roundtrip_0.5_

In [5]:
analysis['data_quality']['stream_snapshot_collapse_case']

{'changed_trade_times': 270585,
 'cumulative_volume_increment': 2890489,
 'day': '2026-08-26',
 'distinct_last_trade_fingerprints': 270760,
 'fingerprint_repeat_quantiles': [1.0, 6.0, 50.0, 10214.0],
 'interpretation': 'LEVELONE_OPTIONS persists state snapshots. Total volume often jumps by multiple contracts while LAST_SIZE describes only the final observed print; unchanged last-trade fields are also repeated across quote updates.',
 'last_size_sum_on_volume_updates': 646953,
 'last_size_to_volume_increment_ratio': 0.22382129805717996,
 'rows_carrying_those_fingerprints': 1527380,
 'stream_snapshots': 1557127,
 'updates_where_volume_delta_exceeds_last_size': 0.6104725934247871,
 'volume_delta_quantiles': [3.0, 24.0, 106.0],
 'volume_updates': 270592}

In [6]:
analysis['flow_date_permutation_placebo']

{'actual_mean_15m_short_points': 0.06227074235806764,
 'method': "permute complete intraday flow profiles across dates while retaining each date's SPX path",
 'one_sided_randomization_p': 0.02594810379241517,
 'placebo_ci95': [-1.22855013698633, 0.03996310593915564],
 'placebo_mean': -0.6083616692010118,
 'samples': 500}

In [7]:
case = analysis['motivating_day_case_study']
print('bearish', [(x['time_et'], x['spx'], x['forward_15m_points']) for x in case.get('bearish_divergences', [])])
print('bullish exits', [(x['time_et'], x['spx'], x['forward_15m_points']) for x in case.get('bullish_exhaustions', [])])

bearish [('09:57', 7685.31, 4.350000000000364), ('10:51', 7675.38, -5.800000000000182), ('11:46', 7678.8, 11.430000000000291), ('12:58', 7664.86, -4.0), ('13:20', 7672.47, -3.3299999999999272), ('13:56', 7675.25, 1.3699999999998909), ('15:03', 7688.64, -0.6099999999996726)]
bullish exits [('10:23', 7675.15, -3.6099999999996726), ('11:18', 7675.77, -3.7700000000004366), ('11:35', 7672.01, 7.119999999999891), ('11:57', 7673.87, -3.5399999999999636), ('12:40', 7660.12, 0.569999999999709), ('14:19', 7675.45, 0.47000000000025466)]


## Takeaways

1. 没有同时通过置信区间、安慰剂和配对策略门；不应接入生产决策。
2. 10:00以后延长持有期没有自动解决稳定性问题；应同时看分段结果和session-block区间。
3. 图上的两类背离可以用现有数据做近似复现，但当前只能称 captured-flow proxy。
4. 若未来要进入方向价差决策，必须先采集完整逐笔/条件码，随后以前向会话验证，再叠加 exact BBO 做 SPXW PnL。

In [8]:
assert analysis['data_quality']['future_rows_used'] == 0
assert analysis['contract']['automatic_ordering'] is False
print('causality and authority checks passed')

causality and authority checks passed
